# Import & Dependency Checker — Live Agent

Run each cell top-to-bottom. Each cell prints ✓ or ✗ for every import.  
**Cell 6** tests `kiteconnect` — it may take 5–15 seconds on first run (SSL init). That is expected.  
If any cell shows ✗ install the missing package with `venv\\Scripts\\python -m pip install <package>`.

In [ ]:
# ── Cell 0: Bootstrap — add project root to sys.path ─────────────────────────
import sys, os
from pathlib import Path

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == 'notebooks':
    PROJECT_ROOT = PROJECT_ROOT.parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))
os.chdir(PROJECT_ROOT)

import platform
print(f'Python  : {sys.version}')
print(f'Platform: {platform.platform()}')
print(f'Root    : {PROJECT_ROOT}')
print()

def _check(label, fn):
    try:
        result = fn()
        ver = f'  ({result})' if result else ''
        print(f'  ✓  {label}{ver}')
        return True
    except Exception as e:
        print(f'  ✗  {label}  →  {e}')
        return False

In [ ]:
# ── Cell 1: Standard library ──────────────────────────────────────────────────
print('=== Standard library ===')
_check('ctypes',    lambda: __import__('ctypes'))
_check('hashlib',   lambda: __import__('hashlib'))
_check('json',      lambda: __import__('json'))
_check('logging',   lambda: __import__('logging'))
_check('threading', lambda: __import__('threading'))
_check('datetime',  lambda: __import__('datetime'))
_check('pathlib',   lambda: __import__('pathlib'))
_check('argparse',  lambda: __import__('argparse'))
_check('time',      lambda: __import__('time'))
print('  done')

In [ ]:
# ── Cell 2: Third-party packages ─────────────────────────────────────────────
print('=== Third-party packages ===')
_check('pandas',       lambda: __import__('pandas').__version__)
_check('numpy',        lambda: __import__('numpy').__version__)
_check('pyarrow',      lambda: __import__('pyarrow').__version__)
_check('requests',     lambda: __import__('requests').__version__)
_check('python-dotenv',lambda: __import__('dotenv').__version__  if hasattr(__import__('dotenv'),'__version__') else 'ok')
_check('pytz',         lambda: __import__('pytz').__version__)
_check('pandas_ta',    lambda: __import__('pandas_ta').__version__)
_check('tqdm',         lambda: __import__('tqdm').__version__)
_check('IPython',      lambda: __import__('IPython').__version__)
print('  done')

In [ ]:
# ── Cell 3: Project config ────────────────────────────────────────────────────
print('=== config.settings ===')
_check('config.settings',         lambda: __import__('config.settings'))

from config.settings import (
    WEIGHTS_FILE, CHECKPOINT_DIR, STOCKS_DIR, INDEX_DIR,
    TRADE_LOG_DIR, UNIVERSE_FILE, AGREEMENT_MIN_LIFETIME_WR
)
print(f'  STOCKS_DIR       : {STOCKS_DIR}')
print(f'  STOCKS_DIR exists: {STOCKS_DIR.exists()}')
print(f'  WEIGHTS_FILE     : {WEIGHTS_FILE}  exists={WEIGHTS_FILE.exists()}')
print(f'  UNIVERSE_FILE    : {UNIVERSE_FILE}  exists={UNIVERSE_FILE.exists()}')
print(f'  CHECKPOINT_DIR   : {CHECKPOINT_DIR}  exists={CHECKPOINT_DIR.exists()}')
print(f'  AGREEMENT_MIN_WR : {AGREEMENT_MIN_LIFETIME_WR}')

In [ ]:
# ── Cell 4: Backtester + strategy imports ─────────────────────────────────────
print('=== backtester ===')
_check('backtester.engine._best_signal',
        lambda: __import__('backtester.engine', fromlist=['_best_signal'])._best_signal)
_check('backtester.engine._conviction_multiplier',
        lambda: __import__('backtester.engine', fromlist=['_conviction_multiplier'])._conviction_multiplier)
_check('backtester.engine._predicted_win_pct',
        lambda: __import__('backtester.engine', fromlist=['_predicted_win_pct'])._predicted_win_pct)
_check('backtester.engine._strategies_fired',
        lambda: __import__('backtester.engine', fromlist=['_strategies_fired'])._strategies_fired)
_check('backtester.engine._LIFETIME_WR',
        lambda: type(__import__('backtester.engine', fromlist=['_LIFETIME_WR'])._LIFETIME_WR).__name__)
_check('backtester.composite_scorer',
        lambda: __import__('backtester.composite_scorer', fromlist=['composite_score']).composite_score)
_check('backtester.quality_filter',
        lambda: __import__('backtester.quality_filter', fromlist=['passes_all_filters']).passes_all_filters)
_check('backtester.position_sizer',
        lambda: __import__('backtester.position_sizer', fromlist=['position_size']).position_size)

print()
print('=== strategies ===')
_check('strategies.ALL_STRATEGIES',
        lambda: f'{len(__import__("strategies", fromlist=["ALL_STRATEGIES"]).ALL_STRATEGIES)} strategies')
_check('weights.regime.get_regime_modifiers',
        lambda: __import__('weights.regime', fromlist=['get_regime_modifiers']).get_regime_modifiers)
_check('watchlist.pre_filter.PreMarketFilter',
        lambda: __import__('watchlist.pre_filter', fromlist=['PreMarketFilter']).PreMarketFilter)

In [ ]:
# ── Cell 5: live/ package (no kiteconnect) ────────────────────────────────────
print('=== live package (kiteconnect-free) ===')
_check('live.candle_builder.CandleBuilder',
        lambda: __import__('live.candle_builder', fromlist=['CandleBuilder']).CandleBuilder)
_check('live.paper_logger.log_closed_trade',
        lambda: __import__('live.paper_logger', fromlist=['log_closed_trade']).log_closed_trade)

# instrument_map no longer imports kiteconnect at module level after the fix
_check('live.instrument_map.NIFTY50_TOKEN',
        lambda: str(__import__('live.instrument_map', fromlist=['NIFTY50_TOKEN']).NIFTY50_TOKEN))

# data_manager uses candle_builder and instrument_map — should be clean
_check('live.data_manager.LiveDataManager',
        lambda: __import__('live.data_manager', fromlist=['LiveDataManager']).LiveDataManager)

# live_engine — imports from backtester (no kiteconnect)
_check('live.live_engine.scan_once',
        lambda: __import__('live.live_engine', fromlist=['scan_once']).scan_once)

# agent module-level import (kiteconnect is deferred to main() and instrument_map fix applied)
_check('live.agent (module import, no kiteconnect)',
        lambda: __import__('live.agent', fromlist=['main']).main)

print()
print('All live/ imports done — if all ✓, Cell 3 of run_live_daily.ipynb will work without hanging.')

In [ ]:
# ── Cell 6: kiteconnect — this may take 5-15 seconds on Python 3.13 ───────────
#
# Expected behaviour:
#   ✓  kiteconnect.KiteConnect   → ok
#   ✓  kiteconnect.KiteTicker    → ok
#   If it hangs here for >60 seconds: Python 3.13 SSL/WebSocket init bug.
#   Workaround: imports are already deferred in agent.py and instrument_map.py.
#   This cell is ONLY a diagnostic — the live agent does NOT import kiteconnect
#   until after you paste your access_token in Cell 2 of run_live_daily.ipynb.
print('=== kiteconnect (may be slow on Python 3.13) ===')
print('Importing KiteConnect...')
_check('kiteconnect.KiteConnect', lambda: __import__('kiteconnect', fromlist=['KiteConnect']).KiteConnect)
print('Importing KiteTicker...')
_check('kiteconnect.KiteTicker',  lambda: __import__('kiteconnect', fromlist=['KiteTicker']).KiteTicker)
print()
print('kiteconnect import complete.')

In [ ]:
# ── Cell 7: Smoke-test CandleBuilder (no network) ─────────────────────────────
#
# day_volume is Kite's CUMULATIVE day volume (monotonically increasing).
# Bar 1 starts from 0 (fresh session)  → bar_vol = vol_at_close - 0
# Bar 2 starts from bar1's close vol   → bar_vol = vol_at_close - vol_at_bar1_close
from live.candle_builder import CandleBuilder
from datetime import datetime

cb = CandleBuilder('TEST')

# ── Bar 1: cumulative day_vol 1000 → 1200 → 1400 ─────────────────────────────
cb.on_tick(100.0, 1000)
cb.on_tick(102.0, 1200)
cb.on_tick(101.0, 1400)
bar1 = cb.close_bar(datetime(2026, 6, 9, 9, 20))

assert bar1 is not None,        'bar1: close_bar returned None'
assert bar1['open']  == 100.0,  f'bar1 open wrong:  {bar1["open"]}'
assert bar1['high']  == 102.0,  f'bar1 high wrong:  {bar1["high"]}'
assert bar1['low']   == 100.0,  f'bar1 low wrong:   {bar1["low"]}'
assert bar1['close'] == 101.0,  f'bar1 close wrong: {bar1["close"]}'
# first bar: vol_start=0 (fresh builder), vol_now=1400 → bar_vol=1400
assert bar1['volume'] == 1400,  f'bar1 volume wrong: {bar1["volume"]}  (expected 1400 = 1400-0)'
print(f'✓  Bar 1: O={bar1["open"]}  H={bar1["high"]}  L={bar1["low"]}  C={bar1["close"]}  V={bar1["volume"]}')

# ── Bar 2: cumulative day_vol 1600 → 1800 (delta = 400) ───────────────────────
cb.on_tick(103.0, 1600)
cb.on_tick(105.0, 1800)
bar2 = cb.close_bar(datetime(2026, 6, 9, 9, 25))

assert bar2 is not None,        'bar2: close_bar returned None'
assert bar2['open']  == 103.0,  f'bar2 open wrong:  {bar2["open"]}'
assert bar2['high']  == 105.0,  f'bar2 high wrong:  {bar2["high"]}'
assert bar2['close'] == 105.0,  f'bar2 close wrong: {bar2["close"]}'
# second bar: vol_start=1400 (carried from bar1), vol_now=1800 → bar_vol=400
assert bar2['volume'] == 400,   f'bar2 volume wrong: {bar2["volume"]}  (expected 400 = 1800-1400)'
print(f'✓  Bar 2: O={bar2["open"]}  H={bar2["high"]}  L={bar2["low"]}  C={bar2["close"]}  V={bar2["volume"]}')

# ── Empty bar (no ticks) ──────────────────────────────────────────────────────
bar3 = cb.close_bar(datetime(2026, 6, 9, 9, 30))
assert bar3 is None, f'empty bar should return None, got {bar3}'
print('✓  Bar 3 (no ticks): correctly returned None')

print()
print('=== All checks complete ===')
print('If all cells show ✓ you are ready to run notebooks/run_live_daily.ipynb')